# 🎯 Shooting Target Recognition — Training Notebook

Полный пайплайн обучения **YOLOv8s** для распознавания пробоин в мишенях.
Модель детектирует пробоины (один класс `bulletholes`); определение зон очков выполняется в Mac-приложении геометрически (Hough Circles).

---

## Шаги
| # | Шаг | Описание |
|---|-----|----------|
| 0 | **Setup** | Установка зависимостей, проверка GPU |
| 1 | **Config** | API-ключ, список источников, гиперпараметры |
| 2 | **Datasets** | Скачать **3 датасета** + объединить в один (1 класс) |
| 3 | **Train** | Обучить YOLOv8s |
| 4 | **Evaluate** | Метрики качества (mAP, Recall, Precision) |
| 5 | **Report** | HTML-отчёт с графиками |
| 6 | **Export** | Сохранить `best.pt` на Google Drive |

> ⚡ **Перед запуском:** `Runtime → Change runtime type → T4 GPU`

## ⚙️ Step 0 — Setup

In [ ]:
# Install dependencies
import subprocess, sys

packages = [
    'ultralytics>=8.3',
    'roboflow>=1.1',
    'opencv-python-headless>=4.8',
    'pandas>=2.0',
    'pyyaml>=6.0',
    'matplotlib>=3.7',
    'seaborn>=0.13',
]
print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('✓ All packages installed')

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu_name}  ({gpu_mem:.1f} GB VRAM)')
else:
    print('⚠  No GPU detected.')
    print('   Go to: Runtime → Change runtime type → T4 GPU')

print(f'PyTorch version: {torch.__version__}')

## 🔑 Step 1 — Configuration

> Отредактируйте эту ячейку перед запуском остальных.

In [ ]:
import os

# ════════════════════════════════════════════════════════════
#  Roboflow API key
#  Получить: https://roboflow.com → Account → API keys
ROBOFLOW_API_KEY = ''   # ← ВСТАВЬТЕ КЛЮЧ СЮДА
# ════════════════════════════════════════════════════════════

# ── Datasets to download and MERGE into one (single class "bulletholes") ─
# Each entry has a per-source `remap` table: source_class → 'bulletholes' or None.
# None means the class will be DROPPED (not a hole — e.g. Target, black_contour).
# All kept classes collapse to class_id=0 ("bulletholes").
#
# version='latest' → use newest available version on Roboflow.
# If a specific version doesn't exist, code auto-falls back to latest with warning.
DATASETS = [
    {
        'name':      'knsa',
        'workspace': 'knsashootingtargets',
        'project':   'knsa-shooting-target',
        'version':   'latest',
        'remap': {
            # KNSA v10+ uses score-zone class taxonomy: 0-10 numeric + 'bulletholes'.
            '0':  'bulletholes', '1':  'bulletholes', '2':  'bulletholes',
            '3':  'bulletholes', '4':  'bulletholes', '5':  'bulletholes',
            '6':  'bulletholes', '7':  'bulletholes', '8':  'bulletholes',
            '9':  'bulletholes', '10': 'bulletholes',
            'bulletholes': 'bulletholes',
        },
    },
    {
        'name':      'project_bat',
        'workspace': 'project-bat-bullet-hole-detection',
        'project':   'bullet-hole-object-detection',
        'version':   'latest',
        'remap': {
            # BAT v30+ uses score-zone class taxonomy: 0-10 numeric + Target + black_contour.
            '0':  'bulletholes', '1':  'bulletholes', '2':  'bulletholes',
            '3':  'bulletholes', '4':  'bulletholes', '5':  'bulletholes',
            '6':  'bulletholes', '7':  'bulletholes', '8':  'bulletholes',
            '9':  'bulletholes', '10': 'bulletholes',
            'Target':        None,    # not a hole — drop
            'black_contour': None,    # not a hole — drop
        },
    },
]

# Training hyperparameters
EPOCHS      = 150
BATCH       = 16      # 32 на A100 / 8 при ограниченной памяти
IMGSZ       = 640
MODEL_SIZE  = 'yolov8s.pt'   # n=fastest, s=balanced, m=accurate
PATIENCE    = 40             # early stopping

# Augmentation (angle invariance — don't reduce without reason)
DEGREES     = 30.0
MOSAIC      = 1.0
MIXUP       = 0.15

# Google Drive
SAVE_TO_DRIVE    = True
DRIVE_MODEL_PATH = '/content/drive/MyDrive/shooting_recognition/best.pt'

# Paths
WORK_DIR = '/content/shooting_target'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Auto-adjust batch for GPU memory
if device == 'cuda':
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if mem_gb < 8:
        BATCH = min(BATCH, 8)
        print(f'⚠  Low VRAM ({mem_gb:.1f} GB) — batch reduced to {BATCH}')

print('Configuration:')
print(f'  Model     : {MODEL_SIZE}')
print(f'  Epochs    : {EPOCHS}  (early stop patience={PATIENCE})')
print(f'  Batch     : {BATCH}')
print(f'  Device    : {device}')
print(f'  Work dir  : {WORK_DIR}')
print(f'  Datasets  : {len(DATASETS)} sources to merge (all using LATEST versions)')
for d in DATASETS:
    print(f"              • {d['name']:<14s} → {d['workspace']}/{d['project']} ({d['version']})")
if not ROBOFLOW_API_KEY:
    print('\n⚠  ROBOFLOW_API_KEY is empty — set it before Step 2!')

In [ ]:
# Mount Google Drive (optional — skip if SAVE_TO_DRIVE = False)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(os.path.dirname(DRIVE_MODEL_PATH), exist_ok=True)
    print(f'✓ Drive mounted. Model will be saved to:\n  {DRIVE_MODEL_PATH}')
else:
    print('Drive not mounted. Model will be lost on runtime reset.')

## 📥 Step 2 — Datasets (download + merge)

Скачиваются все источники из списка `DATASETS` (Step 1) и объединяются в **один датасет** с единственным классом `bulletholes` (`class_id = 0`).

| Источник | Workspace | Версия | ~Изображений |
|---|---|---|---|
| **knsa** | knsashootingtargets/knsa-shooting-target | v8 | ~250 |
| **project_bat** | project-bat-bullet-hole-detection/bullet-hole-object-detection | v1 | ~1240 |
| **andrew_crook** | andrew-crook/bullet-hole | v1 | ~140 |

Каждому источнику сопоставлена `remap`-таблица: hole-классы → `bulletholes`, не-hole классы (Target, black_contour) → `None` (отбрасываются).

Файлы префиксуются именем источника (`knsa_*`, `project_bat_*`, `andrew_crook_*`) для защиты от коллизий имён.

**Выход:** объединённый датасет в формате YOLOv8 в каталоге `data/merged/`.

In [ ]:
import shutil
import yaml
from pathlib import Path
from collections import defaultdict
from roboflow import Roboflow

if not ROBOFLOW_API_KEY:
    raise ValueError('❌ ROBOFLOW_API_KEY is empty. Set it in Step 1.')

RAW_DIR    = Path(WORK_DIR) / 'data' / 'raw'
MERGED_DIR = Path(WORK_DIR) / 'data' / 'merged'
RAW_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

_SPLITS = ('train', 'valid', 'test')
_VALID_ALIASES = ('valid', 'val')   # Roboflow uses either


def resolve_version(proj, requested):
    """Pick a concrete version: 'latest'/None → newest; specific → that or fallback."""
    versions = proj.versions()
    if not versions:
        raise RuntimeError('No versions available for this project')
    available = sorted(int(v.version.split('/')[-1]) for v in versions)
    latest = available[-1]
    if requested in (None, 'latest'):
        return latest
    if int(requested) in available:
        return int(requested)
    print(f'  [WARN] Requested version {requested} not found.')
    print(f'  [WARN] Available versions: {available}. Falling back to latest: {latest}')
    return latest


def download_one(api_key, src):
    """Download a single dataset; return location, or None on failure (skip)."""
    name = src['name']
    location = RAW_DIR / name
    if (location / 'data.yaml').exists():
        print(f'  [{name}] already downloaded → {location}')
        return str(location)
    try:
        rf = Roboflow(api_key=api_key)
        proj = rf.workspace(src['workspace']).project(src['project'])
        version = resolve_version(proj, src.get('version', 'latest'))
        print(f"  [{name}] downloading {src['workspace']}/{src['project']} v{version} …")
        ds = proj.version(version).download(
            model_format='yolov8',
            location=str(location),
            overwrite=False,
        )
        return ds.location
    except Exception as exc:
        print(f'  [ERROR] Failed to download {name}: {exc}')
        print(f'  [INFO]  Skipping this source — merge continues with others.')
        return None


def ingest_source(src_root, src_yaml, remap, prefix):
    """Copy images and rewrite labels into MERGED_DIR. Returns per-split counts."""
    with open(src_yaml) as f:
        cfg = yaml.safe_load(f)
    src_class_names = cfg.get('names', [])

    keep_ids = {}   # src_class_id → 0 (always 'bulletholes')
    for idx, cls_name in enumerate(src_class_names):
        if cls_name in remap and remap[cls_name] is not None:
            keep_ids[idx] = 0

    # Sanity check: warn loudly if no classes matched
    if not keep_ids:
        print(f'  ⚠⚠⚠  CRITICAL: no classes from "{prefix}" matched the remap table!')
        print(f'        Source classes: {src_class_names}')
        print(f'        Remap keys:     {list(remap.keys())}')
        print(f'        ALL annotations from this source will be DROPPED.')

    counts = {sp: 0 for sp in _SPLITS}
    annotations_kept = 0
    for split in _SPLITS:
        src_split = None
        for alias in (_VALID_ALIASES if split == 'valid' else (split,)):
            if (Path(src_root) / alias / 'images').exists():
                src_split = Path(src_root) / alias
                break
        if src_split is None:
            continue

        out_img = MERGED_DIR / split / 'images'
        out_lbl = MERGED_DIR / split / 'labels'

        for img_path in (src_split / 'images').iterdir():
            if not img_path.is_file():
                continue
            new_name = f'{prefix}_{img_path.name}'
            shutil.copy2(img_path, out_img / new_name)

            lbl_src = src_split / 'labels' / (img_path.stem + '.txt')
            new_lines = []
            if lbl_src.exists():
                with open(lbl_src) as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split()
                        try:
                            cid = int(parts[0])
                        except (ValueError, IndexError):
                            continue
                        if cid in keep_ids:
                            parts[0] = str(keep_ids[cid])
                            new_lines.append(' '.join(parts))
            with open(out_lbl / f'{prefix}_{img_path.stem}.txt', 'w') as f:
                f.write('\n'.join(new_lines))
            counts[split] += 1
            annotations_kept += len(new_lines)

    print(f'  [INFO] {prefix}: {sum(counts.values())} images, {annotations_kept} annotations preserved')
    if annotations_kept == 0 and sum(counts.values()) > 0:
        print(f'  ⚠⚠⚠  ZERO annotations kept — training will not work!')
    return counts


# Pre-create merged split dirs
for sp in _SPLITS:
    (MERGED_DIR / sp / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / sp / 'labels').mkdir(parents=True, exist_ok=True)

# ── Run the merge ───────────────────────────────────────────────────────────
MERGED_YAML = MERGED_DIR / 'data.yaml'
if MERGED_YAML.exists():
    print(f'Merged dataset already present → {MERGED_YAML}')
    print('(delete data/merged/ to rebuild from scratch)')
else:
    print(f'Merging {len(DATASETS)} datasets into {MERGED_DIR} …\n')
    successful = 0
    for src in DATASETS:
        loc = download_one(ROBOFLOW_API_KEY, src)
        if loc is None:
            continue
        per_split = ingest_source(
            src_root=loc,
            src_yaml=Path(loc) / 'data.yaml',
            remap=src['remap'],
            prefix=src['name'],
        )
        successful += 1
        print(f"  ✓ {src['name']:<14s}: train={per_split['train']:4d}  "
              f"valid={per_split['valid']:3d}  test={per_split['test']:3d}")

    if successful == 0:
        raise RuntimeError('All sources failed — check API key and dataset names above.')

    # Write merged data.yaml with absolute paths
    abs_out = MERGED_DIR.resolve()
    with open(MERGED_YAML, 'w') as f:
        yaml.dump({
            'train': str(abs_out / 'train' / 'images'),
            'val':   str(abs_out / 'valid' / 'images'),
            'test':  str(abs_out / 'test'  / 'images'),
            'nc':    1,
            'names': ['bulletholes'],
        }, f, allow_unicode=True, sort_keys=False)
    print(f'\n✓ {successful}/{len(DATASETS)} sources merged successfully')

DATA_YAML = str(MERGED_YAML.resolve())
DATA_LOCATION = str(MERGED_DIR.resolve())

# Re-read for downstream cells
with open(MERGED_YAML) as f:
    cfg_yaml = yaml.safe_load(f)

print(f'\n✓ Merged data.yaml → {DATA_YAML}')
print(f'  Classes ({cfg_yaml["nc"]}): {cfg_yaml["names"]}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
from pathlib import Path
from collections import defaultdict

# ── Per-split totals ────────────────────────────────────────────
print(f"Classes ({cfg_yaml['nc']}): {cfg_yaml['names']}")
print()
totals = {}
for split in _SPLITS:
    img_dir = MERGED_DIR / split / 'images'
    n = len(list(img_dir.glob('*.*'))) if img_dir.exists() else 0
    totals[split] = n
    bar = '█' * (n // 8)
    print(f'  {split:6s}: {n:4d} images  {bar}')

# ── Per-source breakdown (count files by prefix) ───────────────
print('\n  Source contributions:')
src_breakdown = defaultdict(lambda: defaultdict(int))
for split in _SPLITS:
    for f in (MERGED_DIR / split / 'images').glob('*'):
        prefix = f.name.split('_', 1)[0]
        src_breakdown[prefix][split] += 1

for src_name, by_split in src_breakdown.items():
    total = sum(by_split.values())
    print(f"    {src_name:<14s}  train={by_split['train']:4d}  "
          f"valid={by_split['valid']:3d}  test={by_split['test']:3d}  "
          f"(total {total})")

# ── Sanity check on size requirements ──────────────────────────
MIN_IMAGES = {'train': 200, 'valid': 50, 'test': 50}
print('\n  Sanity check:')
for sp, min_n in MIN_IMAGES.items():
    ok = '✓' if totals[sp] >= min_n else '⚠ '
    print(f'    {ok}  {sp:6s}: {totals[sp]} images (recommended ≥ {min_n})')

# ── Sample images from MERGED dataset ──────────────────────────
train_imgs = (
    list((MERGED_DIR / 'train' / 'images').glob('*.jpg')) +
    list((MERGED_DIR / 'train' / 'images').glob('*.png')) +
    list((MERGED_DIR / 'train' / 'images').glob('*.jpeg'))
)

if train_imgs:
    samples = random.sample(train_imgs, min(6, len(train_imgs)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor='#0f1117')
    fig.suptitle('Sample training images (merged dataset)', color='white', fontsize=14)
    for ax, img_path in zip(axes.flat, samples):
        img = mpimg.imread(str(img_path))
        ax.imshow(img); ax.axis('off')
        ax.set_title(img_path.name[:28], color='#9ca3af', fontsize=8)
    for ax in axes.flat[len(samples):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No training images found to display.')

## 🏋️ Step 3 — Training

Обучение YOLOv8s с тяжёлой аугментацией для работы под любым углом камеры.  
⏱ ~40–60 мин на T4 GPU при 100 эпохах.

In [ ]:
from ultralytics import YOLO

RUN_DIR = Path(WORK_DIR) / 'runs' / 'train' / 'shooting_target'
BEST_PT = str(RUN_DIR / 'weights' / 'best.pt')

print(f'Training: {MODEL_SIZE} | {EPOCHS} epochs | batch={BATCH} | imgsz={IMGSZ}')
print('─' * 60)

model = YOLO(MODEL_SIZE)
model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    device      = device,
    project     = str(Path(WORK_DIR) / 'runs' / 'train'),
    name        = 'shooting_target',
    exist_ok    = True,
    patience    = PATIENCE,
    save        = True,
    save_period = 10,
    plots       = True,
    # ── Augmentation: angle, scale, colour ───────────────────
    degrees     = DEGREES,
    translate   = 0.15,
    scale       = 0.60,
    shear       = 5.0,
    perspective = 0.0005,
    flipud      = 0.10,
    fliplr      = 0.50,
    mosaic      = MOSAIC,
    mixup       = MIXUP,
    copy_paste  = 0.10,
    hsv_h       = 0.015,
    hsv_s       = 0.70,
    hsv_v       = 0.40,
    # ── Optimiser ────────────────────────────────────────────
    optimizer       = 'AdamW',
    lr0             = 0.001,
    lrf             = 0.01,
    warmup_epochs   = 3,
    weight_decay    = 0.0005,
)

print(f'\n✓ Training complete!')
print(f'  Best weights → {BEST_PT}')

In [ ]:
# ── Training curves ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_files = [
    ('results.png',                    'Loss & Metrics'),
    ('confusion_matrix_normalized.png','Confusion Matrix'),
    ('PR_curve.png',                   'Precision-Recall'),
    ('F1_curve.png',                   'F1 Curve'),
]
available = [(label, RUN_DIR / fname)
             for fname, label in plot_files
             if (RUN_DIR / fname).exists()]

if available:
    cols = min(2, len(available))
    rows = (len(available) + 1) // 2
    fig, axes = plt.subplots(rows, cols,
                             figsize=(10 * cols, 6 * rows),
                             facecolor='#0f1117')
    axes = [axes] if len(available) == 1 else list(
        axes.flat if hasattr(axes, 'flat') else axes)
    for ax, (label, path) in zip(axes, available):
        img = mpimg.imread(str(path))
        ax.imshow(img); ax.axis('off')
        ax.set_title(label, color='white', fontsize=12)
    for ax in axes[len(available):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No plot files found — run training first.')

## 📊 Step 4 — Evaluation

Оценка качества на тестовой выборке.

| Метрика | Цель | Описание |
|---|---|---|
| **mAP@50** | > 0.80 | Основной показатель качества |
| **mAP@50-95** | > 0.55 | Строгая метрика локализации |
| **Precision** | > 0.80 | Доля правильных детекций |
| **Recall** | > 0.75 | Доля найденных пробоин (приоритет) |

In [ ]:
import json
from ultralytics import YOLO

CONF_THRESHOLD = 0.50

print(f'Evaluating on test split | conf={CONF_THRESHOLD}')
eval_model  = YOLO(BEST_PT)
metrics     = eval_model.val(
    data    = DATA_YAML,
    imgsz   = IMGSZ,
    split   = 'test',
    device  = device,
    conf    = CONF_THRESHOLD,
    iou     = 0.50,
    plots   = True,
    verbose = False,
)

map50   = float(metrics.box.map50)
map5095 = float(metrics.box.map)
prec    = float(metrics.box.mp)
recall  = float(metrics.box.mr)

THRESHOLDS = {
    'mAP@50':    (0.80, 0.60),
    'mAP@50-95': (0.55, 0.40),
    'Precision':  (0.80, 0.65),
    'Recall':     (0.75, 0.55),
}

def status(v, g, o):
    return '✓ GOOD' if v >= g else ('~ OK' if v >= o else '✗ LOW')

print('\n══ Overall metrics ══════════════════════════════════════')
for name, val in [('mAP@50', map50), ('mAP@50-95', map5095),
                   ('Precision', prec), ('Recall', recall)]:
    g, o = THRESHOLDS[name]
    bar  = '█' * int(val * 20)
    print(f'  {name:<14} {val:.4f}  {bar:<20} {status(val, g, o)}')

# Per-class
names    = eval_model.names
per_class = {}
if metrics.box.maps is not None:
    print('\n── Per-class mAP@50 ─────────────────────────────────────')
    for i, ap in enumerate(metrics.box.maps):
        cls = names.get(i, str(i))
        per_class[cls] = float(ap)
        bar = '█' * int(ap * 20)
        print(f'  {cls:<14} {ap:.4f}  {bar}')
print('═════════════════════════════════════════════════════════')

# Save metrics
EVAL_RESULTS = {'map50': map50, 'map5095': map5095,
                'precision': prec, 'recall': recall,
                'per_class': per_class}
(RUN_DIR / 'eval_metrics.json').write_text(json.dumps(EVAL_RESULTS, indent=2))
print(f'\n✓ Metrics saved → {RUN_DIR}/eval_metrics.json')

In [ ]:
# ── Per-class bar chart ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if per_class:
    # Sort: numeric score classes first (1-10), then others
    def sort_key(x):
        try: return (0, int(x[0]))
        except: return (1, x[0])

    sorted_cls = sorted(per_class.items(), key=sort_key)
    labels = [c for c, _ in sorted_cls]
    values = [v for _, v in sorted_cls]
    colors = ['#4ade80' if v >= 0.80 else '#fb923c' if v >= 0.60 else '#f87171'
              for v in values]

    fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0f1117')
    ax.set_facecolor('#1a1d27')
    bars = ax.bar(labels, values, color=colors, edgecolor='#2a2d3a', linewidth=0.8)
    ax.axhline(0.80, color='#4ade80', linestyle='--', linewidth=1, label='Target 0.80')
    ax.axhline(0.60, color='#fb923c', linestyle='--', linewidth=1, label='OK 0.60')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', color='white', fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_xlabel('Class (scoring zone)', color='#9ca3af')
    ax.set_ylabel('mAP@50', color='#9ca3af')
    ax.set_title('Per-class mAP@50', color='white', fontsize=13)
    ax.tick_params(colors='#9ca3af')
    ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a', labelcolor='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#2a2d3a')
    plt.tight_layout(); plt.show()
else:
    print('No per-class data available.')

## 📄 Step 5 — Training Report

Генерация HTML-отчёта со всеми метриками и графиками (встроены как base64).

In [ ]:
import base64, csv, datetime
from pathlib import Path
from IPython.display import HTML, display

def _embed(path):
    if not Path(path).exists(): return ''
    data = base64.b64encode(Path(path).read_bytes()).decode()
    return f'<img src="data:image/png;base64,{data}" style="max-width:100%;border-radius:8px;border:1px solid #2a2d3a;margin:4px">'

def _tag(v, g, o):
    if v >= g: return '<span style="background:#16a34a30;color:#4ade80;padding:2px 9px;border-radius:10px;font-weight:700">✓ Good</span>'
    if v >= o: return '<span style="background:#d9770630;color:#fb923c;padding:2px 9px;border-radius:10px;font-weight:700">~ OK</span>'
    return '<span style="background:#dc262630;color:#f87171;padding:2px 9px;border-radius:10px;font-weight:700">✗ Low</span>'

# Metrics (prefer eval step results)
try:
    m = EVAL_RESULTS
except NameError:
    m = {'map50': 0.0, 'map5095': 0.0, 'precision': 0.0, 'recall': 0.0, 'per_class': {}}

thresholds = [
    ('mAP@50',    m['map50'],     0.80, 0.60, '&gt; 0.80'),
    ('mAP@50-95', m['map5095'],   0.55, 0.40, '&gt; 0.55'),
    ('Precision',  m['precision'], 0.80, 0.65, '&gt; 0.80'),
    ('Recall',     m['recall'],    0.75, 0.55, '&gt; 0.75'),
]
assessment = ''.join(
    f"<tr><td>{n}</td><td>{v:.4f}</td><td>{t}</td><td>{_tag(v,g,o)}</td></tr>"
    for n,v,g,o,t in thresholds
)

per_cls_rows = ''.join(
    f"<tr><td>{cls}</td><td>{ap:.4f}</td>"
    f"<td><div style='background:#374151;border-radius:4px;height:10px;width:160px'>"
    f"<div style='background:#3b82f6;height:100%;width:{int(ap*100)}%;border-radius:4px'></div></div></td></tr>"
    for cls, ap in sorted(m.get('per_class', {}).items(),
                           key=lambda x: (0,int(x[0])) if x[0].isdigit() else (1,x[0]))
) or "<tr><td colspan='3'>No per-class data. Run Step 4 first.</td></tr>"

plot_fnames = ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']
plots_html  = ''.join(_embed(RUN_DIR / f) for f in plot_fnames)

html = f'''
<div style="font-family:sans-serif;background:#0f1117;color:#e0e0e0;padding:24px;border-radius:10px">
<h2 style="color:#facc15;margin-top:0">🎯 Training Report — Shooting Target Detection</h2>
<p style="color:#6b7280">{MODEL_SIZE} &nbsp;·&nbsp; {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}</p>

<h3 style="color:#60a5fa">Key Metrics</h3>
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:20px">
  {''.join(f"<div style='background:#1a1d27;border:1px solid #2a2d3a;border-radius:8px;padding:14px;text-align:center'><div style='font-size:2rem;font-weight:700;color:#facc15'>{v:.3f}</div><div style='font-size:.8rem;color:#9ca3af'>{n}</div></div>" for n,v,*_ in thresholds)}
</div>

<h3 style="color:#60a5fa">Assessment</h3>
<table style="border-collapse:collapse;width:100%;margin-bottom:16px">
<tr style="color:#9ca3af;font-size:.8rem"><th style="padding:8px;text-align:left">Metric</th><th>Value</th><th>Target</th><th>Status</th></tr>
{assessment}
</table>

<h3 style="color:#60a5fa">Per-Class mAP@50</h3>
<table style="border-collapse:collapse;width:100%;margin-bottom:16px">
<tr style="color:#9ca3af;font-size:.8rem"><th style="padding:8px;text-align:left">Class</th><th>mAP@50</th><th>Bar</th></tr>
{per_cls_rows}
</table>

<h3 style="color:#60a5fa">Training Curves</h3>
<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">{plots_html}</div>
</div>
'''

REPORT_PATH = RUN_DIR / 'training_report.html'
REPORT_PATH.write_text(html, encoding='utf-8')
print(f'✓ Report saved → {REPORT_PATH}')

# Display inline in notebook
display(HTML(html))

## 💾 Step 6 — Export Model

Сохранить `best.pt` на Google Drive и скачать локально.

In [ ]:
import shutil
from pathlib import Path

best_path = Path(BEST_PT)

if not best_path.exists():
    print(f'❌ best.pt not found at {BEST_PT}')
    print('   Run Step 3 (Training) first.')
else:
    size_mb = best_path.stat().st_size / 1e6
    print(f'✓ best.pt found  ({size_mb:.1f} MB)')

    # Save to Google Drive
    if SAVE_TO_DRIVE:
        drive_path = Path(DRIVE_MODEL_PATH)
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(best_path), str(drive_path))
        print(f'✓ Saved to Drive → {drive_path}')

    # Also save report
    if SAVE_TO_DRIVE and REPORT_PATH.exists():
        drive_report = drive_path.parent / 'training_report.html'
        shutil.copy2(str(REPORT_PATH), str(drive_report))
        print(f'✓ Report to Drive → {drive_report}')

    # Download to local machine
    print('\nDownloading best.pt to your computer...')
    from google.colab import files
    files.download(str(best_path))
    print('✓ Download started')

print(f'\n── Next step ───────────────────────────────────────────')
print(f'Copy best.pt to the Mac app:')
print(f'  shooting_recognition/recognition/weights/best.pt')

---
## 📋 Summary

После успешного обучения:

1. **`best.pt`** скачан и сохранён на Google Drive
2. Скопируйте файл в Mac-приложение:
   ```
   shooting_recognition/recognition/weights/best.pt
   ```
3. Запустите приложение — модель подхватится автоматически:
   ```bash
   python main.py
   ```

### Критерии качества модели
| Метрика | Цель | Что делать если ниже |
|---|---|---|
| mAP@50 | > 0.80 | Добавить эпох или перейти на yolov8m |
| Recall | > 0.75 | Снизить `conf` порог до 0.40 |
| AP класса "10" | может быть низким | Норма — самая маленькая зона |